### Data Analytics Module 1 - Getting Comfortable

### Loading CSV Dataset into Unity Catalog Volumes

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# List of files to download
files = ["2019.csv", "2020.csv", "2021.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DatabricksGenAIEngineer/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading csv files into a dataframe

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv', format='csv')
display(df.limit(100))

### Defining Schema for the Dataframe

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
orderSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*.csv', format='csv', schema=orderSchema)
display(df.limit(100))
     

### Query Data using SQL

In [0]:
df.createOrReplaceTempView("salesorders")
spark_df = spark.sql("SELECT * FROM salesorders")
display(spark_df)

In [0]:
sqlQuery = "SELECT CAST(YEAR(OrderDate) AS CHAR(4)) AS OrderYear, \
               SUM((UnitPrice * Quantity) + Tax) AS GrossRevenue \
        FROM salesorders \
        GROUP BY CAST(YEAR(OrderDate) AS CHAR(4)) \
        ORDER BY OrderYear"
df_spark = spark.sql(sqlQuery)
df_spark.show()

### Using Matplotlib for visualisation

In [0]:
from matplotlib import pyplot as plt

# matplotlib requires a Pandas dataframe, not a Spark one
df_sales = df_spark.toPandas()
# Create a bar plot of revenue by year
plt.bar(x=df_sales['OrderYear'], height=df_sales['GrossRevenue'])
# Display the plot
plt.show()

### Using Seaborn Library for visualisation

In [0]:
import seaborn as sns

# Clear the plot area
plt.clf()
# Create a bar chart
ax = sns.barplot(x="OrderYear", y="GrossRevenue", data=df_sales)
plt.show()